<a href="https://colab.research.google.com/github/rabiaavc/NLP_Notes_Projects/blob/cuce/BGE_M3_Question_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA sürümü:", torch.version.cuda)
print("GPU kullanılabilir mi?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU modeli:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA sürümü: 13.0
GPU kullanılabilir mi? True
GPU modeli: NVIDIA L4


In [ ]:
!nvidia-smi

Mon Nov 16 15:49:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

!pip install -q -U sentence-transformers


## BGE-M3 modelini yükleyelim

In [ ]:

from sentence_transformers import SentenceTransformer
import torch

# BGE-M3 modelini GPU üzerinde yükle
model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cuda"
)

print("Model başarıyla yüklendi!")
print("Kullanılan cihaz:", model.device)
print("Embedding boyutu:", model.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model başarıyla yüklendi!
Kullanılan cihaz: cuda:0
Embedding boyutu: 1024


/tmp/ipykernel_2573/2441252973.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding boyutu:", model.get_sentence_embedding_dimension())


## Anlamsal benzerliği test edelim;

In [ ]:

from sentence_transformers import util
import pandas as pd

questions = [
    "CEN405 dersini kim veriyor?",
    "CEN406 hocası kim?",
    "Who teaches Digital Image Processing?",
    "CEN405 kaç AKTS?",
    "Erasmus'a nasıl başvurabilirim?",
    "Erasmus başvurusunu nereden yapabilirim?",
    "Staj başvurusunu nasıl yapabilirim?",
    "Mezun olmak için kaç kredi gerekiyor?"
]

# Soruları vektörlere dönüştür
embeddings = model.encode(
    questions,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

# Kosinüs benzerliği matrisi
similarity = util.cos_sim(embeddings, embeddings)

similarity_df = pd.DataFrame(
    similarity.cpu().numpy(),
    index=questions,
    columns=questions
)

display(similarity_df.round(3))


,CEN405 dersini kim veriyor?,CEN406 hocası kim?,Who teaches Digital Image Processing?,CEN405 kaç AKTS?,Erasmus'a nasıl başvurabilirim?,Erasmus başvurusunu nereden yapabilirim?,Staj başvurusunu nasıl yapabilirim?,Mezun olmak için kaç kredi gerekiyor?
CEN405 dersini kim veriyor?,1.000,0.697,0.580,0.624,0.378,0.393,0.389,0.466
CEN406 hocası kim?,0.697,1.000,0.411,0.520,0.293,0.290,0.281,0.328
Who teaches Digital Image Processing?,0.580,0.411,1.000,0.244,0.386,0.402,0.385,0.430
CEN405 kaç AKTS?,0.624,0.520,0.244,1.000,0.275,0.270,0.274,0.406
Erasmus'a nasıl başvurabilirim?,0.378,0.293,0.386,0.275,1.000,0.940,0.729,0.571
Erasmus başvurusunu nereden yapabilirim?,0.393,0.290,0.402,0.270,0.940,1.000,0.728,0.550
Staj başvurusunu nasıl yapabilirim?,0.389,0.281,0.385,0.274,0.729,0.728,1.000,0.612
Mezun olmak için kaç kredi gerekiyor?,0.466,0.328,0.430,0.406,0.571,0.550,0.612,1.000


## dataseti yükleyelim

In [ ]:

import pandas as pd
from google.colab import files

# Orijinal veri setini yükle
uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_json(filename, lines=True)

# Orijinal ID, soru ve cevapları koru
df["source_row"] = range(1, len(df) + 1)

# Türkçe etiketli soruları ayır
df_tr = df[
    df["language"].str.lower() == "turkish"
].copy()

print("Toplam kayıt:", len(df))
print("Türkçe kayıt:", len(df_tr))
print("İngilizce kayıt:", len(df) - len(df_tr))

display(df_tr.head())


Saving merged_qa_data.jsonl to merged_qa_data.jsonl
Toplam kayıt: 22524
Türkçe kayıt: 11187
İngilizce kayıt: 11337


,id,question,answer,language,source_row
0,1,Bu dönem Digital Image Processing dersini kim ...,Bu dönem Digital Image Processing (CEN403) der...,turkish,1
8,9,Bu dönem Numerical Analysis dersini kim anlata...,Bu dönem Numerical Analysis (CEN204) dersini k...,turkish,9
24,25,Nesne Yönelimli Programlama Lab dersini kim ve...,Nesne Yönelimli Programlama Lab (CEN216) dersi...,turkish,25
25,26,Nesne Yönelimli Programlama Lab hocası kim?,Nesne Yönelimli Programlama Lab (CEN216) hocas...,turkish,26
26,27,Nesne Yönelimli Programlama Lab dersinin öğret...,Nesne Yönelimli Programlama Lab (CEN216) öğret...,turkish,27


## 500 soruluk pilot veri oluşturma

In [ ]:

# İlk 200 Türkçe soru
first_part = df_tr.head(200)

# Diğer bölümlerden rastgele 300 Türkçe soru
random_part = df_tr.iloc[200:].sample(
    n=300,
    random_state=42
)

# Birleştir
pilot_df = pd.concat([
    first_part,
    random_part
]).sort_values("source_row").reset_index(drop=True)

print("Pilot veri seti:", pilot_df.shape)

display(
    pilot_df[
        ["id", "question", "answer"]
    ].head(10)
)


Pilot veri seti: (500, 5)


,id,question,answer
0,1,Bu dönem Digital Image Processing dersini kim ...,Bu dönem Digital Image Processing (CEN403) der...
1,9,Bu dönem Numerical Analysis dersini kim anlata...,Bu dönem Numerical Analysis (CEN204) dersini k...
2,25,Nesne Yönelimli Programlama Lab dersini kim ve...,Nesne Yönelimli Programlama Lab (CEN216) dersi...
3,26,Nesne Yönelimli Programlama Lab hocası kim?,Nesne Yönelimli Programlama Lab (CEN216) hocas...
4,27,Nesne Yönelimli Programlama Lab dersinin öğret...,Nesne Yönelimli Programlama Lab (CEN216) öğret...
5,28,Kim Nesne Yönelimli Programlama Lab dersini an...,Nesne Yönelimli Programlama Lab (CEN216) dersi...
6,29,Nesne Yönelimli Programlama Lab dersi hangi ho...,Nesne Yönelimli Programlama Lab (CEN216) dersi...
7,30,Nesne Yönelimli Programlama Lab dersinin sorum...,Nesne Yönelimli Programlama Lab (CEN216) dersi...
8,31,Nesne Yönelimli Programlama Lab profesörü kimdir?,Nesne Yönelimli Programlama Lab (CEN216) profe...
9,32,Nesne Yönelimli Programlama Lab dersini hangi ...,Nesne Yönelimli Programlama Lab (CEN216) dersi...


## BGE-M3 Embedding oluşturma

In [ ]:

import numpy as np

# Pilot veri setini kontrol et
print("Soru sayısı:", len(pilot_df))

# Soruları listeye dönüştür
questions = pilot_df["question"].fillna("").astype(str).tolist()

# BGE-M3 ile embedding oluştur
embeddings = model.encode(
    questions,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)
print("Veri tipi:", embeddings.dtype)

# Embeddingleri kaydet
np.save("pilot_bge_m3_embeddings.npy", embeddings)

print("Embeddingler başarıyla kaydedildi!")


Soru sayısı: 500


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding shape: (500, 1024)
Veri tipi: float32
Embeddingler başarıyla kaydedildi!


In [ ]:

from sklearn.cluster import AgglomerativeClustering
import pandas as pd

thresholds = [0.25, 0.35, 0.45]

results = []
cluster_labels = {}

for threshold in thresholds:

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=threshold,
        metric="cosine",
        linkage="average"
    )

    labels = clustering.fit_predict(embeddings)

    cluster_labels[threshold] = labels

    sizes = pd.Series(labels).value_counts()

    results.append({
        "Threshold": threshold,
        "Küme Sayısı": len(sizes),
        "En Büyük Küme": sizes.max(),
        "Tek Soruluk Kümeler": (sizes == 1).sum()
    })

results_df = pd.DataFrame(results)

display(results_df)


,Threshold,Küme Sayısı,En Büyük Küme,Tek Soruluk Kümeler
0,0.25,251,15,190
1,0.35,155,69,89
2,0.45,65,222,29


In [ ]:

# 0.35 eşiğinin sonuçlarını incele
pilot_df["cluster_id"] = cluster_labels[0.35]

# En büyük 5 kümeyi bul
top_clusters = (
    pilot_df["cluster_id"]
    .value_counts()
    .head(5)
)

for cluster_id, count in top_clusters.items():

    print(f"\n{'='*55}")
    print(f"KÜME {cluster_id} | {count} soru")
    print(f"{'='*55}")

    examples = pilot_df[
        pilot_df["cluster_id"] == cluster_id
    ][["id", "question"]].head(10)

    display(examples)



KÜME 1 | 69 soru


,id,question
2,25,Nesne Yönelimli Programlama Lab dersini kim ve...
3,26,Nesne Yönelimli Programlama Lab hocası kim?
4,27,Nesne Yönelimli Programlama Lab dersinin öğret...
5,28,Kim Nesne Yönelimli Programlama Lab dersini an...
6,29,Nesne Yönelimli Programlama Lab dersi hangi ho...
7,30,Nesne Yönelimli Programlama Lab dersinin sorum...
8,31,Nesne Yönelimli Programlama Lab profesörü kimdir?
9,32,Nesne Yönelimli Programlama Lab dersini hangi ...
10,33,Object Oriented Programming Lab dersini kim an...
12,35,Object Oriented Programming Lab dersi kim tara...



KÜME 2 | 47 soru


,id,question
0,1,Bu dönem Digital Image Processing dersini kim ...
32,85,İnternet Programlama dersini kim veriyor?
33,86,İnternet Programlama hocası kim?
34,87,İnternet Programlama dersinin öğretim görevlis...
35,88,Kim İnternet Programlama dersini anlatıyor?
36,89,İnternet Programlama dersi hangi hoca tarafınd...
37,90,İnternet Programlama dersinin sorumlusu kim?
38,91,İnternet Programlama profesörü kimdir?
39,92,İnternet Programlama dersini hangi öğretim üye...
40,93,Internet Programming dersini kim anlatıyor?



KÜME 32 | 34 soru


,id,question
17,55,Sinyaller ve Sistemler dersini kim veriyor?
18,56,Sinyaller ve Sistemler hocası kim?
19,57,Sinyaller ve Sistemler dersinin öğretim görevl...
20,58,Kim Sinyaller ve Sistemler dersini anlatıyor?
21,59,Sinyaller ve Sistemler dersi hangi hoca tarafı...
22,60,Sinyaller ve Sistemler dersinin sorumlusu kim?
23,61,Sinyaller ve Sistemler profesörü kimdir?
24,62,Sinyaller ve Sistemler dersini hangi öğretim ü...
25,63,Signals and Systems dersini kim anlatıyor?
27,65,Signals and Systems dersi kim tarafından okutu...



KÜME 48 | 25 soru


,id,question
11,34,Object Oriented Programming Lab hocasının ismi...
14,37,Object Oriented Programming Lab hocası hakkınd...
26,64,Signals and Systems hocasının ismini nereden ö...
29,67,Signals and Systems hocası hakkında bilgi alab...
56,124,Data Structures Lab hocasının ismini nereden ö...
59,127,Data Structures Lab hocası hakkında bilgi alab...
71,154,Digital Design Lab hocasının ismini nereden öğ...
74,157,Digital Design Lab hocası hakkında bilgi alabi...
86,184,Data Management and File Structures hocasının ...
89,187,Data Management and File Structures hocası hak...



KÜME 11 | 20 soru


,id,question
350,11427,Yaz okulu kapsamında almak istediğim ders konu...
351,11451,Can I take courses from other departments duri...
352,11455,"ACİL! Yaz okulu başlıyor, başka bölümden ders ..."
353,11514,"Ben daha önce arkadaşlarımdan duydum, yaz döne..."
354,11515,Summer school'da interdisciplinary course alab...
355,11522,Yaz okulunda başka bölümden ders alma konusund...
356,11542,"Selam, yaz okulunda farklı bölümlerden ders se..."
365,12246,"Selam, yaz okulu için başka şehirdeki üniversi..."
366,12253,Hazırlık okuyorum ve yaz okulu için araştırma ...
367,12278,"Selam, yaz okulunda online ders alacağım da me..."


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:

# 0.35 eşiğinin sonuçlarını incele
pilot_df["cluster_id"] = cluster_labels[0.35]

# En büyük 5 kümeyi bul
top_clusters = (
    pilot_df["cluster_id"]
    .value_counts()
    .head(5)
)

for cluster_id, count in top_clusters.items():

    print(f"\n{'='*55}")
    print(f"KÜME {cluster_id} | {count} soru")
    print(f"{'='*55}")

    examples = pilot_df[
        pilot_df["cluster_id"] == cluster_id
    ][["id", "question"]].head(10)

    display(examples)



KÜME 1 | 69 soru


,id,question
2,25,Nesne Yönelimli Programlama Lab dersini kim ve...
3,26,Nesne Yönelimli Programlama Lab hocası kim?
4,27,Nesne Yönelimli Programlama Lab dersinin öğret...
5,28,Kim Nesne Yönelimli Programlama Lab dersini an...
6,29,Nesne Yönelimli Programlama Lab dersi hangi ho...
7,30,Nesne Yönelimli Programlama Lab dersinin sorum...
8,31,Nesne Yönelimli Programlama Lab profesörü kimdir?
9,32,Nesne Yönelimli Programlama Lab dersini hangi ...
10,33,Object Oriented Programming Lab dersini kim an...
12,35,Object Oriented Programming Lab dersi kim tara...



KÜME 2 | 47 soru


,id,question
0,1,Bu dönem Digital Image Processing dersini kim ...
32,85,İnternet Programlama dersini kim veriyor?
33,86,İnternet Programlama hocası kim?
34,87,İnternet Programlama dersinin öğretim görevlis...
35,88,Kim İnternet Programlama dersini anlatıyor?
36,89,İnternet Programlama dersi hangi hoca tarafınd...
37,90,İnternet Programlama dersinin sorumlusu kim?
38,91,İnternet Programlama profesörü kimdir?
39,92,İnternet Programlama dersini hangi öğretim üye...
40,93,Internet Programming dersini kim anlatıyor?



KÜME 32 | 34 soru


,id,question
17,55,Sinyaller ve Sistemler dersini kim veriyor?
18,56,Sinyaller ve Sistemler hocası kim?
19,57,Sinyaller ve Sistemler dersinin öğretim görevl...
20,58,Kim Sinyaller ve Sistemler dersini anlatıyor?
21,59,Sinyaller ve Sistemler dersi hangi hoca tarafı...
22,60,Sinyaller ve Sistemler dersinin sorumlusu kim?
23,61,Sinyaller ve Sistemler profesörü kimdir?
24,62,Sinyaller ve Sistemler dersini hangi öğretim ü...
25,63,Signals and Systems dersini kim anlatıyor?
27,65,Signals and Systems dersi kim tarafından okutu...



KÜME 48 | 25 soru


,id,question
11,34,Object Oriented Programming Lab hocasının ismi...
14,37,Object Oriented Programming Lab hocası hakkınd...
26,64,Signals and Systems hocasının ismini nereden ö...
29,67,Signals and Systems hocası hakkında bilgi alab...
56,124,Data Structures Lab hocasının ismini nereden ö...
59,127,Data Structures Lab hocası hakkında bilgi alab...
71,154,Digital Design Lab hocasının ismini nereden öğ...
74,157,Digital Design Lab hocası hakkında bilgi alabi...
86,184,Data Management and File Structures hocasının ...
89,187,Data Management and File Structures hocası hak...



KÜME 11 | 20 soru


,id,question
350,11427,Yaz okulu kapsamında almak istediğim ders konu...
351,11451,Can I take courses from other departments duri...
352,11455,"ACİL! Yaz okulu başlıyor, başka bölümden ders ..."
353,11514,"Ben daha önce arkadaşlarımdan duydum, yaz döne..."
354,11515,Summer school'da interdisciplinary course alab...
355,11522,Yaz okulunda başka bölümden ders alma konusund...
356,11542,"Selam, yaz okulunda farklı bölümlerden ders se..."
365,12246,"Selam, yaz okulu için başka şehirdeki üniversi..."
366,12253,Hazırlık okuyorum ve yaz okulu için araştırma ...
367,12278,"Selam, yaz okulunda online ders alacağım da me..."


In [ ]:

# Küme 1 ve 11'i ayrıntılı inceleyelim

for cid in [1, 11]:
    group = pilot_df[
        pilot_df["cluster_id"] == cid
    ]

    print("\n" + "=" * 70)
    print(f"KÜME {cid} | {len(group)} soru")
    print("=" * 70)

    for _, row in group.iterrows():
        print(f"ID {row['id']}: {row['question']}")



KÜME 1 | 69 soru
ID 25: Nesne Yönelimli Programlama Lab dersini kim veriyor?
ID 26: Nesne Yönelimli Programlama Lab hocası kim?
ID 27: Nesne Yönelimli Programlama Lab dersinin öğretim görevlisi kimdir?
ID 28: Kim Nesne Yönelimli Programlama Lab dersini anlatıyor?
ID 29: Nesne Yönelimli Programlama Lab dersi hangi hoca tarafından veriliyor?
ID 30: Nesne Yönelimli Programlama Lab dersinin sorumlusu kim?
ID 31: Nesne Yönelimli Programlama Lab profesörü kimdir?
ID 32: Nesne Yönelimli Programlama Lab dersini hangi öğretim üyesi veriyor?
ID 33: Object Oriented Programming Lab dersini kim anlatıyor?
ID 35: Object Oriented Programming Lab dersi kim tarafından okutulacak?
ID 36: Object Oriented Programming Lab dersinin akademisyeni kim?
ID 38: Object Oriented Programming Lab dersinin koordinatörü kimdir?
ID 39: Bu dönem Object Oriented Programming Lab dersini kim anlatacak?
ID 115: Veri Yapıları Lab dersini kim veriyor?
ID 116: Veri Yapıları Lab hocası kim?
ID 117: Veri Yapıları Lab dersinin ö

In [ ]:

!pip -q install hdbscan

import hdbscan
import numpy as np
import pandas as pd

# Daha önce oluşturduğumuz BGE-M3 embeddingleri
X = embeddings.astype(np.float64)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom"
)

labels_hdbscan = clusterer.fit_predict(X)

pilot_df["hdbscan_cluster"] = labels_hdbscan

# Sonuçları hesapla
n_clusters = len(set(labels_hdbscan) - {-1})
n_noise = np.sum(labels_hdbscan == -1)

print("HDBSCAN Sonuçları")
print("---------------------------")
print("Toplam soru:", len(pilot_df))
print("Oluşan küme sayısı:", n_clusters)
print("Kümelenemeyen soru:", n_noise)
print("Kümelenen soru:", len(pilot_df) - n_noise)

print("\nEn büyük 10 küme:")
print(
    pilot_df.loc[
        pilot_df["hdbscan_cluster"] != -1,
        "hdbscan_cluster"
    ].value_counts().head(10).to_string()
)


HDBSCAN Sonuçları
---------------------------
Toplam soru: 500
Oluşan küme sayısı: 26
Kümelenemeyen soru: 237
Kümelenen soru: 263

En büyük 10 küme:
hdbscan_cluster
5     40
14    17
6     17
13    14
15    13
7     13
18    13
8     12
10    12
25    10


In [ ]:

# HDBSCAN'in en büyük iki kümesini bul
top_clusters = (
    pilot_df.loc[
        pilot_df["hdbscan_cluster"] != -1,
        "hdbscan_cluster"
    ]
    .value_counts()
    .head(2)
)

for cluster_id, count in top_clusters.items():

    group = pilot_df[
        pilot_df["hdbscan_cluster"] == cluster_id
    ]

    print("\n" + "=" * 65)
    print(f"KÜME {cluster_id} | {count} soru")
    print("=" * 65)

    for _, row in group.iterrows():
        print(f"ID {row['id']}: {row['question']}")



KÜME 5 | 40 soru
ID 6459: Yüksek lisanstan mezun olma sartlari nelerdir?
ID 6588: Yüksek lisans kaç yıl?
ID 6593: Ben daha önce başka bir üniversitede yüksek lisans başlamıştım ama yarıda bıraktım, şimdi tekrar başlayacağım, süre yine 2 yıl mı?
ID 6601: Yüksek lisans süresinin ne kadar olduğunu tam olarak bilmiyorum, 1.5 yıl mı 2 yıl mı?
ID 11200: Yaz okulu dilekçesinde hangi yılların taban puanları kabul ediliyor, sadece kayıt yaptırdığım yıl mı yoksa güncel puanlar da olur mu?
ID 11323: Application process için yaz okulu başvurusu timing'i nasıl olmalı?
ID 11383: Hocam, yaz okulu süreciyle ilgili detaylı bilgi alabilir miyim? Başvuru sonrası kayıt işlemleri tam olarak nasıl ilerliyor?
ID 11392: Yaz okulu sürecinde tam olarak hangi belgeler gerekiyor ve kayıt işlemi için spesifik olarak ne zaman başvurmalıyım?
ID 11427: Yaz okulu kapsamında almak istediğim ders konusunda bilgi almak istiyorum. İlgili ders başka üniversitede farklı bölüm tarafından verilmektedir. Bu durumda dersi alab

In [ ]:

from sklearn.cluster import KMeans
import pandas as pd

# Beklenen eşleşmeler: 1 = aynı niyet, 0 = farklı niyet
test_pairs = [
    (25, 115, 1),        # Farklı dersler, aynı hoca sorgusu
    (25, 85, 1),         # Aynı öğretim üyesi niyeti
    (11451, 11515, 1),   # Başka bölümden yaz okulu dersi
    (12246, 18474, 1),  # Başka şehirden yaz okulu dersi
    (6588, 6593, 1),     # Yüksek lisans süresi
    (11451, 12246, 0),  # Başka bölüm / başka üniversite
    (11451, 12636, 0),  # Ders alma / seçmeli saydırma
    (6459, 11427, 0),   # Yüksek lisans / yaz okulu
    (85, 11451, 0),     # Öğretim üyesi / yaz okulu
    (11451, 12950, 0)   # Başka bölüm / ders sayısı sınırı
]

id_to_idx = {
    int(id_): i
    for i, id_ in enumerate(pilot_df["id"])
}

methods = {
    "Agglomerative 0.35": cluster_labels[0.35],
    "HDBSCAN": labels_hdbscan
}

# K-Means denemeleri
for k in [30, 50, 80]:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    methods[f"K-Means {k}"] = km.fit_predict(embeddings)

# Bütün yöntemleri aynı örneklerle karşılaştır
results = []

for name, labels in methods.items():
    correct = 0
    tested = 0

    for id1, id2, expected in test_pairs:
        if id1 not in id_to_idx or id2 not in id_to_idx:
            continue

        c1 = labels[id_to_idx[id1]]
        c2 = labels[id_to_idx[id2]]

        # -1 HDBSCAN'de kümelenemeyen soru
        same_cluster = int(c1 == c2 and c1 != -1)

        correct += int(same_cluster == expected)
        tested += 1

    results.append({
        "Algoritma": name,
        "Doğru eşleşme": correct,
        "Test sayısı": tested,
        "Başarı (%)": round(100 * correct / tested, 1)
                         if tested else None
    })

display(pd.DataFrame(results))


,Algoritma,Doğru eşleşme,Test sayısı,Başarı (%)
0,Agglomerative 0.35,6,10,60.0
1,HDBSCAN,4,10,40.0
2,K-Means 30,5,10,50.0
3,K-Means 50,6,10,60.0
4,K-Means 80,7,10,70.0


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# BGE-M3 embeddingleriyle benzerlik matrisi
similarity_matrix = cosine_similarity(embeddings)

print("Similarity matrix:", similarity_matrix.shape)

# Orijinal ID -> embedding indeksi
id_to_idx = {
    int(row["id"]): idx
    for idx, row in pilot_df.iterrows()
}

# Aynı ve farklı niyetleri temsil eden soru çiftleri
test_pairs = [
    (25, 115, "Aynı niyet"),
    (25, 85, "Aynı niyet"),
    (11451, 11515, "Aynı niyet"),
    (12246, 18474, "Aynı niyet"),
    (11451, 12246, "Farklı niyet"),
    (11451, 12636, "Farklı niyet"),
    (6459, 11427, "Farklı niyet")
]

results = []

for id1, id2, expected in test_pairs:
    if id1 not in id_to_idx or id2 not in id_to_idx:
        continue

    i, j = id_to_idx[id1], id_to_idx[id2]

    results.append({
        "ID 1": id1,
        "ID 2": id2,
        "Beklenen": expected,
        "Cosine Similarity": round(
            float(similarity_matrix[i, j]), 4
        )
    })

display(pd.DataFrame(results))


Similarity matrix: (500, 500)


,ID 1,ID 2,Beklenen,Cosine Similarity
0,25,115,Aynı niyet,0.7659
1,25,85,Aynı niyet,0.7111
2,11451,11515,Aynı niyet,0.8002
3,12246,18474,Aynı niyet,0.8637
4,11451,12246,Farklı niyet,0.7258
5,11451,12636,Farklı niyet,0.7176
6,6459,11427,Farklı niyet,0.4556


In [ ]:

def find_similar_questions(question_id, top_k=10):
    if question_id not in id_to_idx:
        print("Bu ID pilot veri setinde bulunmuyor.")
        return

    idx = id_to_idx[question_id]
    scores = similarity_matrix[idx].copy()

    # Sorunun kendisini hariç tut
    scores[idx] = -np.inf

    nearest = np.argsort(scores)[::-1][:top_k]

    results = []

    for i in nearest:
        results.append({
            "ID": pilot_df.iloc[i]["id"],
            "Question": pilot_df.iloc[i]["question"],
            "Similarity": round(float(scores[i]), 4)
        })

    display(pd.DataFrame(results))

find_similar_questions(25, top_k=10)


,ID,Question,Similarity
0,32,Nesne Yönelimli Programlama Lab dersini hangi ...,0.9513
1,29,Nesne Yönelimli Programlama Lab dersi hangi ho...,0.9472
2,27,Nesne Yönelimli Programlama Lab dersinin öğret...,0.9369
3,30,Nesne Yönelimli Programlama Lab dersinin sorum...,0.9355
4,28,Kim Nesne Yönelimli Programlama Lab dersini an...,0.9342
5,1490,Nesne Yönelimli Programlama dersini kim veriyor?,0.9322
6,26,Nesne Yönelimli Programlama Lab hocası kim?,0.8873
7,31,Nesne Yönelimli Programlama Lab profesörü kimdir?,0.8732
8,33,Object Oriented Programming Lab dersini kim an...,0.8191
9,39,Bu dönem Object Oriented Programming Lab dersi...,0.8111


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(embeddings)

print("Benzerlik matrisi:", sim.shape)

Benzerlik matrisi: (500, 500)


In [ ]:

import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

thresholds = [0.65, 0.70, 0.72, 0.73, 0.75, 0.80, 0.85, 0.90]

results = []

for threshold in thresholds:
    y_true = []
    y_pred = []

    for id1, id2, expected in test_pairs:
        if id1 not in id_to_idx or id2 not in id_to_idx:
            continue

        score = float(sim[
            id_to_idx[id1],
            id_to_idx[id2]
        ])

        y_true.append(expected)
        y_pred.append(int(score >= threshold))

    results.append({
        "Threshold": threshold,
        "Accuracy": round(accuracy_score(y_true, y_pred), 3),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 3)
    })

results_df = pd.DataFrame(results)

display(results_df)


ValueError: Mix of label input types (string and number)